In [0]:
storage_account = "ecomadlsmsk"

configs = {
    "fs.azure.account.auth.type": "OAuth",

    "fs.azure.account.oauth.provider.type":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",

    "fs.azure.account.oauth2.client.id":
        "089ad881-9047-4677-b43d-e2080e594d6b",

    "fs.azure.account.oauth2.client.secret":
        dbutils.secrets.get(
            scope="ecom-adls",
            key="service-principal-secret"
        ),

    "fs.azure.account.oauth2.client.endpoint":
        "https://login.microsoftonline.com/"
        "0db06a44-42c2-4948-91a6-a1cb03720eb5/oauth2/token"
}

for key, value in configs.items():
    spark.conf.set(
        f"{key}.{storage_account}.dfs.core.windows.net",
        value
    )

In [0]:
#This notebook will just be our bronze layer, we will be importing our data from our container locations and store it as 
# managed tables. 
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *



In [0]:
spark = SparkSession.builder.appName("EcomDataPipeline").getOrCreate()

In [0]:
#Reading the data from the adls locations
userDF = spark.read.format("parquet")\
    .option("header", 'true')\
    .option("inferSchema", 'true')\
    .load("abfss://landing-zone-2@ecomadlsmsk.dfs.core.windows.net/users-raw-2/")


In [0]:
#Write users data to managed table
userDF.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_fashion.bronze.users")

In [0]:
#Similarly we are going to do the same with buyers, sellers and countries data

buyersDF = spark.read.format("parquet")\
    .option("header", 'true')\
    .option("inferschema", 'true')\
    .load("abfss://landing-zone-2@ecomadlsmsk.dfs.core.windows.net/buyers-raw-2/")

In [0]:
buyersDF.show(5)

In [0]:
buyersDF.write \
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("ecommerce_fashion.bronze.buyers")

In [0]:
countriesDF = spark.read.format("parquet")\
    .option("header", 'true')\
    .option("inferschema", 'true')\
    .load("abfss://landing-zone-2@ecomadlsmsk.dfs.core.windows.net/countries-raw-2/")

In [0]:
countriesDF.write \
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("ecommerce_fashion.bronze.countries")

In [0]:
sellersDF = spark.read.format("parquet")\
    .option("inferschema", 'true')\
    .option("header", 'true')\
    .load("abfss://landing-zone-2@ecomadlsmsk.dfs.core.windows.net/sellers-raw-2/")

In [0]:
sellersDF.write \
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("ecommerce_fashion.bronze.sellers")